In [ ]:
from typing import cast

from datasets import load_dataset, Dataset
from vllm import LLM, SamplingParams

In [ ]:
ds = load_dataset("karanravindra/simple-wikipedia")

train_ds = cast(Dataset, ds["train"])
val_ds = cast(Dataset, ds["val"])
test_ds = cast(Dataset, ds["test"])

print(
    f"Dataset Sizes: train={len(train_ds):,} val={len(val_ds):,} test={len(test_ds):,}"
)

In [ ]:
model = LLM(
    "ibm-granite/granite-4.0-h-tiny",
    gpu_memory_utilization=0.8,
    quantization="fp8",
)

In [ ]:
texts = val_ds.filter(lambda x: len(x["tokens"]) < 128_000)["text"]

system_prompt = """You are an expert technical writer.

**Task**:
You will be given a passage of text from `SimpleWikipedia`. Your task is to write a detailed textbook
about the passage. The textbook should be comprehensive and cover all the important aspects of the passage.
It should be written in a clear and concise manner, making it easy for readers to understand the content.
The textbook should include examples, explanations, and any relevant information that can help readers grasp
the concepts presented in the passage.

**Output Format**:
The output should be a well-structured markdown textbook that includes the following sections:
1. Title: A concise and informative title for the textbook.
2. Introduction: A brief introduction to the topic covered in the passage.
3. Main Content: A detailed explanation of the concepts presented in the passage, organized into subsections as needed.
4. Examples: Relevant examples that illustrate the concepts discussed in the main content.
5. Conclusion: A summary of the key points covered in the textbook.

**Guidelines**:
- Ensure that the textbook is comprehensive and covers all important aspects of the passage, as well as anything else that is relevant to the topic.
- Write in a clear and concise manner, making it easy for readers to understand the content.
- Use markdown formatting to structure the textbook and make it visually appealing.
- Avoid including any information that is not relevant to the topic covered in the passage.
- Textbook should be between 500 and 2000 words in length.
"""
prompts = [
    [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": f"Generate a textbook for the following passage:\n\n{text}",
        },
    ]
    for text in texts
]

responses = model.chat(
    prompts, sampling_params=SamplingParams(temperature=0, top_p=0.9, max_tokens=2048)
)

In [ ]:
import re


def wrap_text(text: str, width: int = 80, whitespace_pre_wrap: bool = True) -> str:
    """Wraps text using regex

    Args:
        text (str): The text to wrap
        width (int, optional): The width of the wrap. Defaults to 80.
        whitespace_pre_wrap (bool, optional): Whether to split on word boundaries. Defaults to True.

    Returns:
        str: The wrapped text
    """
    if whitespace_pre_wrap:
        # Split on word boundaries
        pattern = rf"(.{{1,{width}}})(\s+|$)"
    else:
        # Split on any character
        pattern = rf"(.{{1,{width}}})"

    wrapped_text = re.findall(pattern, text)
    return "\n".join([line[0] for line in wrapped_text])


for i, (prompt, output) in enumerate(zip(prompts[:10], responses[:10])):
    # skip if less than 1000 chars
    if len(prompt[1]["content"]) < 1000:
        continue

    print(f"== Example {i + 1:03d} " + "=" * (100 - len(f"== Example {i + 1:03d} ")))
    print(f"-- Original Passage (length={len(prompt[1]['content']):,} chars) ---")
    print(wrap_text(prompt[1]["content"], width=100))
    print("\n")
    print(f"-- Rewritten Passage (length={len(output.outputs[0].text):,} chars) ---")
    print(wrap_text(output.outputs[0].text, width=100))
    print("\n")

In [ ]:
system_prompt = """You are an expert evaluator of language models.

**Task**:
You will be given two passages of text: a reference passage and a generated passage. Your task is to evaluate the quality of the generated passage compared to the reference passage. The evaluation should be based on the following criteria:

1. Relevance: How well does the generated passage cover the important aspects of the reference passage? Does it include all the key points and relevant information?
2. Clarity: Is the generated passage written in a clear and concise manner? Is it easy to understand and follow?
3. Coherence: Is the generated passage well-structured and organized? Does it flow logically from one point to the next?
4. Creativity: Does the generated passage demonstrate creativity in its presentation of the information? Does it provide unique insights or perspectives that are not present in the reference passage?

**Output Format**:
Output in the following format: `<score> <justification>`\

**Output Example**:
```0.8 The generated passage covers most of the key points from the reference passage and includes relevant information. It is written in a clear and concise manner, making it easy to understand. The passage is well-structured and organized, with a logical flow. However, it lacks some creativity and does not provide unique insights or perspectives that are present in the reference passage.
```
"""

eval_responses = model.chat(
    [
        [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": f"Reference Passage:\n\n<doc>\n{p[1]['content']}\n</doc>\n\nGenerated Passage:\n\n<doc>\n{o.outputs[0].text}\n</doc>\n\nEvaluate the quality of the generated passage compared to the reference passage and provide a score between 0 and 1.",
            },
        ]
        for p, o in zip(prompts[:10], responses[:10])
    ],
    sampling_params=SamplingParams(temperature=0.0, top_k=1, max_tokens=10),
)

In [ ]:
[e.outputs[0].text for e in eval_responses[:10]]